# NB03 — MapReduce + Word Count
## Investor Intelligence Platform — FIIs Brasil
### CRISP-DM Phase: Modeling

| Item | Value |
|------|-------|
| **Input** | `data/silver/silver_articles.parquet` |
| **Output** | `data/gold/word_count/*.parquet` + `nb03_report.json` |
| **Sources** | 21: 20 portais editoriais + Reddit |
| **RDD ops** | `parallelize` `flatMap` `map` `reduceByKey` `sortBy` `collect` |
| **Seed** | `RANDOM_SEED=42` · `adaptive.enabled=false` |

> **Entregaveis:** word count global · word count por source · TOFU term frequency · **Desafio do Professor:** termos de alta frequencia em contexto negativo

## SEÇÃO 1 — Imports

Importa PySpark, NLTK para stopwords em português e utilitários de NLP leve para a etapa de tokenização.

In [1]:
import sys, os
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
import os, re, sys, json, random, warnings
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import nltk

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, FloatType, BooleanType
import pyspark

warnings.filterwarnings("ignore")
print(f"Python  : {sys.version.split()[0]}")
print(f"PySpark : {pyspark.__version__}")
print(f"NLTK    : {nltk.__version__}")
print(f"Pandas  : {pd.__version__}")

Python  : 3.12.12
PySpark : 3.5.1
NLTK    : 3.8.1
Pandas  : 2.2.1


## SEÇÃO 2 — Configuração, SparkSession e NLTK

Importa constantes do `config/settings.py` (NB00), define paths Gold, inicia SparkSession com config idêntica a NB01/NB02 e baixa stopwords PT-BR do NLTK.

In [2]:
# -- Project root
PROJECT_ROOT = Path(os.getcwd()).resolve()
_lim = 6
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.parent != PROJECT_ROOT and _lim > 0:
    PROJECT_ROOT = PROJECT_ROOT.parent
    _lim -= 1
sys.path.insert(0, str(PROJECT_ROOT))

import sys, os
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
from config.settings import (
    RANDOM_SEED, SPARK_DRIVER_MEMORY, SPARK_APP_NAME, SPARK_SHUFFLE_PARTS,
)
from src.utils.logger import ingestion_logger, log_quality_check, log_spark_start

# -- Reprodutibilidade
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# -- Paths
SILVER_DIR   = PROJECT_ROOT / "data" / "silver"
GOLD_WC_DIR  = PROJECT_ROOT / "data" / "gold" / "word_count"
GOLD_WC_DIR.mkdir(parents=True, exist_ok=True)

# -- SparkSession (config identica a NB01/NB02)
spark = (
    SparkSession.builder
    .appName(f"{SPARK_APP_NAME}_wc")
    .master("local[*]")
    .config("spark.driver.memory",          SPARK_DRIVER_MEMORY)
    .config("spark.sql.shuffle.partitions", SPARK_SHUFFLE_PARTS)
    .config("spark.sql.adaptive.enabled",   "false")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

logger = ingestion_logger()
log_spark_start(logger, spark.sparkContext.appName, spark.version)

# -- NLTK stopwords PT-BR
for _res in ["stopwords", "punkt"]:
    try:
        nltk.data.find(f"corpora/{_res}")
    except LookupError:
        nltk.download(_res, quiet=True)
from nltk.corpus import stopwords as nltk_sw
STOPWORDS_PT = set(nltk_sw.words("portuguese"))

print(f"PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"SILVER_DIR     : {SILVER_DIR}")
print(f"GOLD_WC_DIR    : {GOLD_WC_DIR}")
print(f"SparkSession   : {spark.sparkContext.appName}")
print(f"Stopwords PT   : {len(STOPWORDS_PT)} termos")

26/06/06 18:00:37 WARN Utils: Your hostname, MacBook-Air-100.local resolves to a loopback address: 127.0.0.1; using 192.168.15.13 instead (on interface en0)
26/06/06 18:00:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/06 18:00:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/06 18:00:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/06 18:00:38 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


2026-06-06 18:00:38 | INFO     | fii.ingestion                | SPARK_START | app=Investor-Intelligence-Platform-FIIs_wc | version=3.5.1
PROJECT_ROOT   : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/__notebooks
SILVER_DIR     : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/__notebooks/data/silver
GOLD_WC_DIR    : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/__notebooks/data/gold/word_count
SparkSession   : Investor-Intelligence-Platform-FIIs_wc
Stopwords PT   : 207 termos


## SEÇÃO 3 — Leitura da Camada Silver

Carrega `silver_articles.parquet` produzido pelo NB02 e valida a presença das colunas obrigatórias para word count: `text_clean`, `source_label`, `source_type`.

In [3]:
SILVER_PATH = SILVER_DIR / "silver_articles.parquet"
if not SILVER_PATH.exists():
    raise FileNotFoundError(
        f"Silver nao encontrado: {SILVER_PATH}\nExecute NB02 primeiro."
    )

df_silver = pd.read_parquet(SILVER_PATH)

# -- Validacao de colunas minimas
_REQ = ["article_id", "text_clean", "source_label", "source_type",
        "word_count", "has_content"]
_miss = [c for c in _REQ if c not in df_silver.columns]
if _miss:
    raise ValueError(f"Colunas ausentes no Silver: {_miss}")

# -- Filtrar apenas registros com conteudo util
df_silver = df_silver[df_silver["has_content"] == True].copy()
df_silver["text_clean"] = df_silver["text_clean"].fillna("").astype(str)

print(f"Silver carregado: {len(df_silver):,} registros uteis")
print(f"  source_type:\n{df_silver['source_type'].value_counts().to_string()}")
print(f"\n  source_label (top 5):\n{df_silver['source_label'].value_counts().head().to_string()}")
print(f"\n  word_count medio : {df_silver['word_count'].mean():.1f}")
print(f"  word_count total : {df_silver['word_count'].sum():,}")

Silver carregado: 122 registros uteis
  source_type:
source_type
scraping    63
rss         59

  source_label (top 5):
source_label
CNN Brasil Business    29
FIIs.com.br            12
Funds Explorer         12
Investidor10           10
XP Conteudos            7

  word_count medio : 456.6
  word_count total : 55,703


## SEÇÃO 4 — Pré-processamento NLP: Stopwords, Termos e Tokenizador

Define stopwords PT-BR + domínio FII, os dicionários de termos TOFU e negativos do briefing, e a função `tokenize()` com regex que preserva acentuação portuguesa.

In [4]:
# -- Stopwords customizadas para dominio FII
_CUSTOM_SW = {
    "ser", "ter", "fazer", "poder", "querer", "haver", "estar",
    "anos", "ano", "meses", "mes", "dias", "dia",
    "vez", "vezes", "forma", "modo", "tipo", "caso", "parte",
    "valor", "valores", "numero", "numeros",
    "ainda", "tambem", "pode", "sendo", "entre", "sobre",
    "desta", "deste", "desta", "disso", "isso", "esse", "essa",
    "vai", "vao", "faz", "fez", "foi", "sao", "sao", "tem", "tем",
    "mais", "menos", "muito", "pouco", "grande", "pequeno",
    "novo", "novos", "nova", "novas", "primeiro", "segunda",
    "real", "reais",   # moeda — ambiguo demais
    "brasil", "brasileiro", "brasileira", "brasileiros",
    "mercado",            # muito generico
}
STOPWORDS = STOPWORDS_PT | _CUSTOM_SW

# -- Termos TOFU (briefing oficial) — unigrams normalizados
TOFU_TERMS = {
    "dividendos", "passiva", "patrimonio", "recorrencia",
    "diversificacao", "yield", "inflacao", "gestao", "carteira",
    "previsibilidade", "proventos", "valorizacao",
    "fluxo", "independencia", "imobiliarios", "mensal",
    "investimento", "fundos", "renda", "caixa",
}

# -- Bigrams TOFU compostos
TOFU_BIGRAMS = {
    "renda_passiva", "renda_mensal", "fluxo_caixa",
    "independencia_financeira", "longo_prazo", "fundos_imobiliarios",
    "dividendos_mensais", "geracao_renda", "valorizacao_patrimonial",
    "carteira_diversificada",
}

# -- Termos negativos (briefing oficial)
NEGATIVE_TERMS = {
    "vacancia", "inadimplencia", "risco", "prejuizo", "corte",
    "alavancagem", "queda", "crise", "desvalorizacao",
    "deterioracao", "incerteza", "liquidez", "reducao",
    "juros", "credito", "perdas", "negativo", "baixa",
}

# -- Normalizador de acentos para lookup
import unicodedata
def _norm(s: str) -> str:
    return unicodedata.normalize("NFD", s).encode("ascii", "ignore").decode("ascii").lower()

TOFU_NORM    = {_norm(t) for t in TOFU_TERMS}
NEGATIVE_NORM= {_norm(t) for t in NEGATIVE_TERMS}

# -- Tokenizador: lowercase + regex PT + stopwords
_TOKEN_RE = re.compile(r"[a-zà-ü]{3,}", re.IGNORECASE | re.UNICODE)

def tokenize(text: str) -> list:
    '''Tokeniza texto PT-BR: lowercase, regex unicode, sem stopwords.'''
    if not text or not isinstance(text, str):
        return []
    tokens = _TOKEN_RE.findall(text.lower())
    return [_norm(t) for t in tokens if _norm(t) not in STOPWORDS and len(t) >= 3]

def tokenize_with_bigrams(text: str) -> list:
    '''Tokeniza + bigrams para deteccao de termos compostos TOFU.'''
    tokens = tokenize(text)
    bigrams = [f"{tokens[i]}_{tokens[i+1]}" for i in range(len(tokens) - 1)]
    return tokens + bigrams

# -- Smoke test
_t = tokenize("Fundos imobiliarios pagam dividendos mensais com alta previsibilidade.")
assert "dividendos" in _t, f"dividendos ausente: {_t}"
assert "imobiliarios" in _t, f"imobiliarios ausente: {_t}"
print(f"tokenize smoke test OK: {_t}")
print(f"Stopwords totais : {len(STOPWORDS)}")
print(f"TOFU terms       : {len(TOFU_NORM)}")
print(f"Negative terms   : {len(NEGATIVE_NORM)}")

tokenize smoke test OK: ['fundos', 'imobiliarios', 'pagam', 'dividendos', 'mensais', 'alta', 'previsibilidade']
Stopwords totais : 259
TOFU terms       : 20
Negative terms   : 18


## SEÇÃO 5 — MapReduce: Word Count Global

Pipeline RDD principal: `parallelize → flatMap → map → reduceByKey → sortBy → collect`. Produz a frequência global de todos os termos no corpus Silver (21 sources).

In [5]:
# -- RDD de textos Silver
_texts = df_silver["text_clean"].dropna().tolist()

rdd_texts = spark.sparkContext.parallelize(_texts)

# ── Pipeline MapReduce principal ─────────────────────────────────────────────
# parallelize -> flatMap (tokenize) -> map (word,1) -> reduceByKey -> sortBy -> collect
word_count_global = (
    rdd_texts
    .flatMap(lambda text: tokenize(text))
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
    .collect()
)

_vocab_size = len(word_count_global)
_total_tokens = sum(c for _, c in word_count_global)
print(f"Word Count Global completo")
print(f"  Vocabulario  : {_vocab_size:,} termos unicos")
print(f"  Total tokens : {_total_tokens:,}")

# -- Top 50
print(f"\n  Top 50 termos:")
print(f"  {'Rank':<6} {'Termo':<25} {'Frequencia':>10} {'TOFU':>6} {'NEG':>5}")
print(f"  {'-'*56}")
for rank, (term, freq) in enumerate(word_count_global[:50], 1):
    is_tofu = "✓" if term in TOFU_NORM else ""
    is_neg  = "✓" if term in NEGATIVE_NORM else ""
    print(f"  {rank:<6} {term:<25} {freq:>10,} {is_tofu:>6} {is_neg:>5}")

Word Count Global completo
  Vocabulario  : 8,119 termos unicos
  Total tokens : 32,669

  Top 50 termos:
  Rank   Termo                     Frequencia   TOFU   NEG
  --------------------------------------------------------
  1      nao                              198             
  2      renda                            174      ✓      
  3      fundos                           164      ✓      
  4      fiis                             153             
  5      fundo                            146             
  6      todos                            126             
  7      acoes                            123             
  8      dividendos                       123      ✓      
  9      risco                            115            ✓
  10     carteira                         112      ✓      
  11     imobiliarios                     107      ✓      
  12     ativos                           100             
  13     ver                               97             
  14     

## SEÇÃO 6 — MapReduce: Word Count por Source

Segundo pipeline RDD: tokeniza por `(source_label, word)` e reduz por chave composta, produzindo ranking de termos por cada um dos 21 portais/Reddit.

In [6]:
# -- RDD de (source_label, text_clean)
_src_texts = (
    df_silver[["source_label", "source_type", "text_clean"]]
    .dropna()
    .values.tolist()
)

rdd_src = spark.sparkContext.parallelize(_src_texts)

# -- Pipeline: (source, word) -> count
wc_per_source_raw = (
    rdd_src
    .flatMap(lambda r: [((r[0], r[1], w), 1) for w in tokenize(r[2])])
    .reduceByKey(lambda a, b: a + b)
    .map(lambda x: (x[0][0], x[0][1], x[0][2], x[1]))
    .sortBy(lambda x: (x[0], -x[3]))
    .collect()
)

# -- DataFrame com rank por source
_rows = []
_current, _rank = None, 0
for source_label, source_type, term, freq in wc_per_source_raw:
    if source_label != _current:
        _current, _rank = source_label, 0
    _rank += 1
    _rows.append({
        "source_label": source_label,
        "source_type":  source_type,
        "term":         term,
        "frequency":    freq,
        "rank_in_source": _rank,
        "is_tofu":     term in TOFU_NORM,
        "is_negative": term in NEGATIVE_NORM,
    })

df_source_wc = pd.DataFrame(_rows)
print(f"Word Count por Source: {len(df_source_wc):,} linhas")
print(f"  Sources cobertos  : {df_source_wc['source_label'].nunique()}")

# -- Top 10 por source (amostra)
print("\n  Top 5 termos por source (primeiros 4 sources):")
_sample_sources = df_source_wc["source_label"].unique()[:4]
for _src in _sample_sources:
    _top = df_source_wc[df_source_wc["source_label"] == _src].head(5)
    print(f"\n  [{_src}]")
    for _, row in _top.iterrows():
        print(f"    {row['rank_in_source']:>3}. {row['term']:<22} {row['frequency']:>6,}")

Word Count por Source: 13,095 linhas
  Sources cobertos  : 17

  Top 5 termos por source (primeiros 4 sources):

  [Bora Investir (B3)]
      1. renda                       8
      2. investimentos               5
      3. acoes                       4
      4. comparador                  4
      5. etfs                        4

  [CNN Brasil Business]
      1. nao                        71
      2. segundo                    49
      3. onde                       30
      4. apos                       29
      5. eclipse                    27

  [Clube FII]
      1. fii                         3
      2. criar                       3
      3. cookies                     3
      4. clube                       3
      5. carteiras                   2

  [E-Investidor Estadao]
      1. nao                        18
      2. berkshire                  13
      3. apple                      13
      4. jobs                       13
      5. buffett                    11


## SEÇÃO 7 — Frequência de Termos TOFU

Filtra o word count global pelos termos TOFU do briefing (unigrams + bigrams compostos), rankeia por frequência e mapeia em quais sources cada termo é mais relevante.

In [7]:
# -- Filtrar termos TOFU do word count global
_wc_dict = dict(word_count_global)

# Bigrams TOFU — novo pipeline RDD
rdd_bigrams = spark.sparkContext.parallelize(_texts)
wc_bigrams = (
    rdd_bigrams
    .flatMap(lambda t: tokenize_with_bigrams(t))
    .filter(lambda w: "_" in w)          # apenas bigrams
    .map(lambda w: (w, 1))
    .reduceByKey(lambda a, b: a + b)
    .filter(lambda x: x[0] in TOFU_BIGRAMS)
    .sortBy(lambda x: x[1], ascending=False)
    .collect()
)
_bigram_dict = dict(wc_bigrams)

# -- DataFrame TOFU unigrams + bigrams
_tofu_rows = []
for term in sorted(TOFU_NORM | TOFU_BIGRAMS):
    freq = _wc_dict.get(term, _bigram_dict.get(term, 0))
    _tofu_rows.append({
        "term":      term,
        "frequency": freq,
        "compound":  "_" in term,
        "present":   freq > 0,
    })

df_tofu = (
    pd.DataFrame(_tofu_rows)
    .sort_values("frequency", ascending=False)
    .reset_index(drop=True)
)
df_tofu["rank"] = df_tofu.index + 1

print("Frequencia de Termos TOFU (briefing oficial):")
print(f"  {'Rank':<5} {'Termo':<30} {'Frequencia':>10} {'Composto':>9}")
print(f"  {'-'*58}")
for _, row in df_tofu[df_tofu["present"]].iterrows():
    print(f"  {int(row['rank']):<5} {row['term']:<30} {int(row['frequency']):>10,} {str(row['compound']):>9}")

_absent = df_tofu[~df_tofu["present"]]["term"].tolist()
if _absent:
    print(f"\n  Termos TOFU ausentes no corpus: {_absent}")

Frequencia de Termos TOFU (briefing oficial):
  Rank  Termo                          Frequencia  Composto
  ----------------------------------------------------------
  1     renda                                 174     False
  2     fundos                                164     False
  3     dividendos                            123     False
  4     carteira                              112     False
  5     imobiliarios                          107     False
  6     fundos_imobiliarios                    86      True
  7     investimento                           79     False
  8     gestao                                 63     False
  9     inflacao                               45     False
  10    longo_prazo                            30      True
  11    diversificacao                         25     False
  12    patrimonio                             24     False
  13    fluxo                                  15     False
  14    mensal                                 14    

## SEÇÃO 8 — Detecção de Termos Negativos no Corpus

Contabiliza frequência e cobertura (nº de artigos) de cada termo negativo do briefing no corpus Silver, por source_type, para entender onde o risco aparece mais.

In [8]:
# -- Para cada termo negativo: frequencia global + artigos que o contem
_neg_rows = []
for neg_term in sorted(NEGATIVE_NORM):
    _freq = _wc_dict.get(neg_term, 0)
    _art_count = df_silver["text_clean"].apply(
        lambda t: neg_term in " ".join(tokenize(t)) if isinstance(t, str) else False
    ).sum()
    _neg_rows.append({
        "negative_term":   neg_term,
        "freq_global":     _freq,
        "articles_with_term": int(_art_count),
        "article_coverage_pct": round(_art_count / len(df_silver) * 100, 2),
    })

df_negative_terms = (
    pd.DataFrame(_neg_rows)
    .sort_values("freq_global", ascending=False)
    .reset_index(drop=True)
)

print("Termos Negativos no Corpus:")
print(f"  {'Termo':<22} {'Freq Global':>12} {'Artigos':>8} {'Cobertura%':>11}")
print(f"  {'-'*57}")
for _, row in df_negative_terms.iterrows():
    print(f"  {row['negative_term']:<22} {int(row['freq_global']):>12,} "
          f"{int(row['articles_with_term']):>8,} {row['article_coverage_pct']:>10.1f}%")

Termos Negativos no Corpus:
  Termo                   Freq Global  Artigos  Cobertura%
  ---------------------------------------------------------
  risco                           115       23       18.9%
  juros                            77       22       18.0%
  liquidez                         57       20       16.4%
  credito                          53       17       13.9%
  queda                            20       10        8.2%
  baixa                            16       12        9.8%
  perdas                            9        8        6.6%
  inadimplencia                     8        3        2.5%
  negativo                          5        3        2.5%
  crise                             4        5        4.1%
  reducao                           4        4        3.3%
  alavancagem                       3        2        1.6%
  corte                             3        8        6.6%
  vacancia                          3        3        2.5%
  incerteza                

## SEÇÃO 9 — Desafio Academico: Termos de Alta Frequência em Contexto Negativo

Filtra artigos com pelo menos um termo negativo e executa um novo pipeline RDD sobre esse sub-corpus para identificar termos que co-ocorrem com conteúdo negativo. Calcula `negative_context_ratio = freq_neg_ctx / freq_global` e classifica o risco.

In [9]:
# -- Artigos com contexto negativo (contem >= 1 termo negativo)
def has_negative(text: str) -> bool:
    '''True se o texto contem pelo menos um termo negativo normalizado.'''
    if not text or not isinstance(text, str):
        return False
    tokens = set(tokenize(text))
    return bool(tokens & NEGATIVE_NORM)

_neg_mask   = df_silver["text_clean"].apply(has_negative)
df_neg_ctx  = df_silver[_neg_mask].copy()
df_pos_ctx  = df_silver[~_neg_mask].copy()

print(f"Artigos com contexto negativo : {len(df_neg_ctx):,} ({len(df_neg_ctx)/len(df_silver)*100:.1f}%)")
print(f"Artigos sem contexto negativo : {len(df_pos_ctx):,} ({len(df_pos_ctx)/len(df_silver)*100:.1f}%)")

# -- Pipeline RDD no sub-corpus negativo
rdd_neg = spark.sparkContext.parallelize(df_neg_ctx["text_clean"].tolist())

wc_neg_ctx = (
    rdd_neg
    .flatMap(lambda t: tokenize(t))
    .map(lambda w: (w, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
    .collect()
)

_wc_neg_dict = dict(wc_neg_ctx)

# -- Calcular negative_context_ratio
_MIN_GLOBAL_FREQ = 5     # ignora hapax legomena
_MIN_NEG_FREQ    = 3

_neg_ctx_rows = []
for term, freq_global in word_count_global:
    if freq_global < _MIN_GLOBAL_FREQ:
        continue
    freq_neg = _wc_neg_dict.get(term, 0)
    if freq_neg < _MIN_NEG_FREQ:
        continue
    ratio = freq_neg / freq_global
    risk  = "ALTO"   if ratio >= 0.60 else \
            "MEDIO"  if ratio >= 0.35 else "BAIXO"
    _neg_ctx_rows.append({
        "term":                 term,
        "freq_global":          freq_global,
        "freq_negative_ctx":    freq_neg,
        "negative_ctx_ratio":   round(ratio, 4),
        "is_tofu":              term in TOFU_NORM,
        "is_negative_term":     term in NEGATIVE_NORM,
        "risk_level":           risk,
    })

df_neg_analysis = (
    pd.DataFrame(_neg_ctx_rows)
    .sort_values(["risk_level", "negative_ctx_ratio"], ascending=[True, False])
    .reset_index(drop=True)
)

print(f"\nAnalise de contexto negativo: {len(df_neg_analysis):,} termos com ratio calculado")
print(f"  ALTO  (ratio >= 0.60): {(df_neg_analysis['risk_level'] == 'ALTO').sum():,}")
print(f"  MEDIO (ratio >= 0.35): {(df_neg_analysis['risk_level'] == 'MEDIO').sum():,}")
print(f"  BAIXO (ratio <  0.35): {(df_neg_analysis['risk_level'] == 'BAIXO').sum():,}")

print("\n  Termos ALTO risco (contexto negativo):")
_alto = df_neg_analysis[df_neg_analysis["risk_level"] == "ALTO"].head(20)
print(f"  {'Termo':<25} {'Freq Global':>12} {'Freq NegCtx':>11} {'Ratio':>7} {'TOFU':>5}")
print(f"  {'-'*65}")
for _, row in _alto.iterrows():
    _tofu_flag = "YES" if row["is_tofu"] else ""
    print(f"  {row['term']:<25} {int(row['freq_global']):>12,} "
          f"{int(row['freq_negative_ctx']):>11,} {row['negative_ctx_ratio']:>7.3f} {_tofu_flag:>5}")

Artigos com contexto negativo : 54 (44.3%)
Artigos sem contexto negativo : 68 (55.7%)

Analise de contexto negativo: 1,354 termos com ratio calculado
  ALTO  (ratio >= 0.60): 1,127
  MEDIO (ratio >= 0.35): 189
  BAIXO (ratio <  0.35): 38

  Termos ALTO risco (contexto negativo):
  Termo                      Freq Global Freq NegCtx   Ratio  TOFU
  -----------------------------------------------------------------
  risco                              115         115   1.000      
  juros                               77          77   1.000      
  liquidez                            57          57   1.000      
  selic                               54          54   1.000      
  credito                             53          53   1.000      
  titulos                             45          45   1.000      
  ipca                                45          45   1.000      
  cdi                                 44          44   1.000      
  taxa                                41         

## SEÇÃO 10 — Construção do DataFrame Gold Consolidado

Constrói o DataFrame Gold principal do NB03 unindo word count global com flags TOFU, negativo e risk_level — base para TF-IDF no NB04 e sentiment no NB05.

In [10]:
# -- Gold: word count global com todas as flags
_global_rows = []
_neg_map = {r["term"]: r for r in _neg_ctx_rows}

for rank, (term, freq) in enumerate(word_count_global, 1):
    _neg_info = _neg_map.get(term, {})
    _global_rows.append({
        "term":                term,
        "frequency":           freq,
        "rank":                rank,
        "is_tofu":             term in TOFU_NORM,
        "is_negative_term":    term in NEGATIVE_NORM,
        "freq_negative_ctx":   _neg_info.get("freq_negative_ctx", 0),
        "negative_ctx_ratio":  _neg_info.get("negative_ctx_ratio", 0.0),
        "risk_level":          _neg_info.get("risk_level", "NA"),
    })

df_global_wc = pd.DataFrame(_global_rows)

print(f"Gold word count global: {len(df_global_wc):,} termos")
print(f"  TOFU terms presentes  : {df_global_wc['is_tofu'].sum()}")
print(f"  Negative terms        : {df_global_wc['is_negative_term'].sum()}")
print(f"  Com risk_level ALTO   : {(df_global_wc['risk_level']=='ALTO').sum()}")

Gold word count global: 8,119 termos
  TOFU terms presentes  : 19
  Negative terms        : 17
  Com risk_level ALTO   : 1127


## SEÇÃO 11 — Escrita da Camada Gold

Escreve quatro parquets na camada Gold (`data/gold/word_count/`) e o relatório JSON de rastreabilidade do NB03, prontos para leitura pelo NB04 e NB05.

In [11]:
_NOW = datetime.now(timezone.utc).isoformat()

# -- 1. Global word count (vocabulario completo)
_p1 = GOLD_WC_DIR / "global_word_count.parquet"
df_global_wc.to_parquet(_p1, index=False, engine="pyarrow")
print(f"[1] {_p1.name}: {len(df_global_wc):,} linhas")

# -- 2. Word count por source
_p2 = GOLD_WC_DIR / "source_word_count.parquet"
df_source_wc.to_parquet(_p2, index=False, engine="pyarrow")
print(f"[2] {_p2.name}: {len(df_source_wc):,} linhas")

# -- 3. TOFU term frequency
_p3 = GOLD_WC_DIR / "tofu_frequency.parquet"
df_tofu.to_parquet(_p3, index=False, engine="pyarrow")
print(f"[3] {_p3.name}: {len(df_tofu):,} linhas")

# -- 4. Negative context analysis
_p4 = GOLD_WC_DIR / "negative_context.parquet"
df_neg_analysis.to_parquet(_p4, index=False, engine="pyarrow")
print(f"[4] {_p4.name}: {len(df_neg_analysis):,} linhas")

# -- Relatorio JSON
_report = {
    "notebook":       "NB03 word_count_mapreduce",
    "timestamp":      _NOW,
    "input_path":     str(SILVER_PATH),
    "output_dir":     str(GOLD_WC_DIR),
    "corpus_stats": {
        "total_articles":    len(df_silver),
        "total_tokens":      int(_total_tokens),
        "vocabulary_size":   int(_vocab_size),
        "neg_ctx_articles":  len(df_neg_ctx),
        "neg_ctx_pct":       round(len(df_neg_ctx)/len(df_silver)*100, 2),
    },
    "outputs": [
        {"file": _p1.name, "rows": len(df_global_wc)},
        {"file": _p2.name, "rows": len(df_source_wc)},
        {"file": _p3.name, "rows": len(df_tofu)},
        {"file": _p4.name, "rows": len(df_neg_analysis)},
    ],
    "rdd_pipelines": [
        "global_word_count  : parallelize -> flatMap -> map -> reduceByKey -> sortBy -> collect",
        "source_word_count  : parallelize -> flatMap -> reduceByKey -> map -> sortBy -> collect",
        "bigram_detection   : parallelize -> flatMap -> filter -> map -> reduceByKey -> collect",
        "negative_ctx_count : parallelize -> flatMap -> map -> reduceByKey -> sortBy -> collect",
    ],
}
_rp = GOLD_WC_DIR / "nb03_report.json"
_rp.write_text(json.dumps(_report, indent=2, ensure_ascii=False))
print(f"\nRelatorio: {_rp}")
print(f"\nGold word_count pronto em: {GOLD_WC_DIR}")

[1] global_word_count.parquet: 8,119 linhas
[2] source_word_count.parquet: 13,095 linhas
[3] tofu_frequency.parquet: 30 linhas
[4] negative_context.parquet: 1,354 linhas

Relatorio: /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/__notebooks/data/gold/word_count/nb03_report.json

Gold word_count pronto em: /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/__notebooks/data/gold/word_count


## SEÇÃO 12 — Checks e Validação dos Outputs Gold

Recarrega os quatro parquets Gold do disco, valida integridade e exibe `df.head(5)` de cada output para confirmação visual.

In [12]:
print("=" * 70)
print("  CHECKS -- GOLD / word_count")
print("=" * 70)

_outputs = {
    "global_word_count.parquet":  ["term", "frequency", "rank", "is_tofu", "risk_level"],
    "source_word_count.parquet":  ["source_label", "term", "frequency", "rank_in_source"],
    "tofu_frequency.parquet":     ["term", "frequency", "compound"],
    "negative_context.parquet":   ["term", "freq_global", "negative_ctx_ratio", "risk_level"],
}

for fname, req_cols in _outputs.items():
    _path = GOLD_WC_DIR / fname
    _df   = pd.read_parquet(_path)
    _miss = [c for c in req_cols if c not in _df.columns]
    _ok   = "OK" if not _miss else f"AVISO: {_miss}"
    print(f"\n  [{_ok}] {fname} — {len(_df):,} linhas")
    print(_df[req_cols].head(5).to_string(index=False))

# -- Resumo critico
print("\n  Resumo:")
_gw = pd.read_parquet(GOLD_WC_DIR / "global_word_count.parquet")
print(f"  Vocabulario total      : {len(_gw):,}")
print(f"  Top 1 termo            : {_gw.iloc[0]['term']} ({int(_gw.iloc[0]['frequency']):,}x)")
print(f"  TOFU terms encontrados : {_gw['is_tofu'].sum()}/{len(TOFU_NORM)}")
_nc = pd.read_parquet(GOLD_WC_DIR / "negative_context.parquet")
print(f"  Termos risco ALTO      : {(_nc['risk_level']=='ALTO').sum()}")
print(f"  Termos risco MEDIO     : {(_nc['risk_level']=='MEDIO').sum()}")
print("\n  Gold/word_count verificado -- pronto para NB04")

  CHECKS -- GOLD / word_count

  [OK] global_word_count.parquet — 8,119 linhas
  term  frequency  rank  is_tofu risk_level
   nao        198     1    False       ALTO
 renda        174     2     True       ALTO
fundos        164     3     True       ALTO
  fiis        153     4    False       ALTO
 fundo        146     5    False       ALTO

  [OK] source_word_count.parquet — 13,095 linhas
      source_label          term  frequency  rank_in_source
Bora Investir (B3)         renda          8               1
Bora Investir (B3) investimentos          5               2
Bora Investir (B3)         acoes          4               3
Bora Investir (B3)    comparador          4               4
Bora Investir (B3)          etfs          4               5

  [OK] tofu_frequency.parquet — 30 linhas
        term  frequency  compound
       renda        174     False
      fundos        164     False
  dividendos        123     False
    carteira        112     False
imobiliarios        107     False


## SEÇÃO 13 — CRISP-DM Traceability e Encerramento

Exibe quadro de rastreabilidade CRISP-DM do NB03 com todos os pipelines RDD executados e encerra o SparkSession.

In [13]:
_box = [
    "=" * 78,
    "  NB03 -- MapReduce + Word Count  |  CRISP-DM: Modeling",
    "=" * 78,
    f"  Input   : {str(SILVER_PATH)}",
    f"  Output  : {str(GOLD_WC_DIR)}",
    "-" * 78,
    "  RDD Pipelines executados:",
    "    1. Global Word Count : parallelize->flatMap->map->reduceByKey->sortBy->collect",
    "    2. Per-Source Count  : parallelize->flatMap->reduceByKey->map->sortBy->collect",
    "    3. Bigram Detection  : parallelize->flatMap->filter->map->reduceByKey->collect",
    "    4. Negative Context  : parallelize->flatMap->map->reduceByKey->sortBy->collect",
    "-" * 78,
    "  Desafio do Academico: termos de alta frequencia em contexto negativo",
    "  Metodo: sub-corpus negativo -> word count -> negative_ctx_ratio",
    "  Risk levels: ALTO (ratio>=0.60) | MEDIO (>=0.35) | BAIXO (<0.35)",
    "-" * 78,
    "  LGPD: corpus publico editorial -- nenhum dado pessoal",
    "  Seed: RANDOM_SEED=42 | adaptive.enabled=false",
    "-" * 78,
    "  Proximo: NB04 -- TF-IDF + BM25  (le data/silver/ + data/gold/word_count/)",
    "=" * 78,
]
print("\n".join(_box))

_gw = pd.read_parquet(GOLD_WC_DIR / "global_word_count.parquet")
print(f"\n  Vocabulario     : {len(_gw):,} termos")
print(f"  Total tokens    : {_total_tokens:,}")
print(f"  Artigos corpus  : {len(df_silver):,}")
print(f"  Artigos neg ctx : {len(df_neg_ctx):,} ({len(df_neg_ctx)/len(df_silver)*100:.1f}%)")
print("\n  NB03 completo -- Gold/word_count pronto para NB04")

spark.stop()
import sys, os
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
print("  SparkSession encerrada")

  NB03 -- MapReduce + Word Count  |  CRISP-DM: Modeling
  Input   : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/__notebooks/data/silver/silver_articles.parquet
  Output  : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/__notebooks/data/gold/word_count
------------------------------------------------------------------------------
  RDD Pipelines executados:
    1. Global Word Count : parallelize->flatMap->map->reduceByKey->sortBy->collect
    2. Per-Source Count  : parallelize->flatMap->reduceByKey->map->sortBy->collect
    3. Bigram Detection  : parallelize->flatMap->filter->map->reduceByKey->collect
    4. Negative Context  : parallelize->flatMap->map->reduceByKey->sortBy->collect
------------------------------------------------------------------------------
  Desafio do Academico: termos de alta frequencia em contexto negativo
  Metodo: sub-corpus negativo -> word count -> negative_ctx_ratio
  Risk levels: ALTO (ratio>=0.60) | ME